In [6]:
import numpy as np
import os
from scipy.special import logsumexp

def compute_expected_bias_vectorized(b, c, a=2, d=3, n_trials=2, n_train=100, n_test=200, n_grid=60):
    np.random.seed(42)

    Y_train = np.random.normal(0, 1, (n_trials, n_train))
    Y_test = np.random.normal(0, 1, (n_trials, n_test))

    t_vals = np.linspace(-1.5, 1.5, n_grid)
    theta1, theta2 = np.meshgrid(t_vals, t_vals)

    mean_s = (theta1.flatten() ** a) * (theta2.flatten() ** b)
    mean_t = (theta1.flatten() ** c) * (theta2.flatten() ** d)

    PCIC_trials = np.zeros(n_trials)
    G_n_trials = np.zeros(n_trials)

    for trial in range(n_trials):
        Y_tr = Y_train[trial]
        Y_te = Y_test[trial]

        log_s_train = -0.5 * ((Y_tr[:, None] - mean_s[None, :])**2) - 0.5 * np.log(2 * np.pi)

        log_post = np.sum(log_s_train, axis=0)
        log_post -= logsumexp(log_post)
        post_probs = np.exp(log_post)

        log_h_train = -0.5 * ((Y_tr[:, None] - mean_t[None, :])**2) - 0.5 * np.log(2 * np.pi)
        log_h_test  = -0.5 * ((Y_te[:, None] - mean_t[None, :])**2) - 0.5 * np.log(2 * np.pi)

        log_E_h_train = logsumexp(log_h_train + log_post[None, :], axis=1)
        log_E_h_test  = logsumexp(log_h_test  + log_post[None, :], axis=1)

        T_n = -np.mean(log_E_h_train)
        G_n = -np.mean(log_E_h_test)
        G_n_trials[trial] = G_n

        E_log_h_train   = np.sum(log_h_train * post_probs[None, :], axis=1)
        E_s_train       = np.sum(log_s_train * post_probs[None, :], axis=1)
        E_log_h_s_train = np.sum(log_h_train * log_s_train * post_probs[None, :], axis=1)

        pcic_pen = np.mean(E_log_h_s_train - E_log_h_train * E_s_train)
        PCIC_trials[trial] = T_n + pcic_pen

    return np.mean(PCIC_trials - G_n_trials), np.mean(np.abs(PCIC_trials - G_n_trials))


def run_simulation_grid():
    a_values = [1, 2, 3, 4, 5]
    c_values = [1, 2, 3, 4, 5]

    degrees = list(range(1, 19))
    n_deg = len(degrees)

    results = {}

    for a in a_values:
        for c in c_values:
            pcic_biases_raw = np.zeros((n_deg, n_deg))
            pcic_biases_abs = np.zeros((n_deg, n_deg))

            for i, b_val in enumerate(degrees):
                for j, d_val in enumerate(degrees):
                    raw, abs_ = compute_expected_bias_vectorized(
                        b=b_val, c=c, a=a, d=d_val
                    )
                    pcic_biases_raw[i, j] = raw
                    pcic_biases_abs[i, j] = abs_

            results[(a, c)] = {"raw": pcic_biases_raw, "abs": pcic_biases_abs}

    return results, degrees, degrees

results, b_vals, d_vals = run_simulation_grid()

In [11]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
import numpy as np
import seaborn as sns
import os

ABS_CMAP = "YlOrRd"

def plot_all(results, b_vals, d_vals, out_dir="./abcd"):
    os.makedirs(out_dir, exist_ok=True)

    all_raw = np.concatenate([v["raw"].ravel() for v in results.values()])
    all_abs = np.concatenate([v["abs"].ravel() for v in results.values()])

    # --- Global norms for grid plots ---
    nonzero_raw = all_raw[all_raw != 0]
    linthresh   = np.percentile(np.abs(nonzero_raw), 10) if len(nonzero_raw) else 1.0
    raw_vabs    = np.percentile(np.abs(all_raw), 95)
    norm_raw    = mcolors.SymLogNorm(linthresh=linthresh, vmin=-raw_vabs, vmax=raw_vabs)

    pos_abs  = all_abs[all_abs > 0]
    abs_vmin = np.percentile(pos_abs, 5)  if len(pos_abs) else 1e-10  # was pos_abs.min()
    abs_vmax = np.percentile(pos_abs, 95) if len(pos_abs) else 1.0
    norm_abs = mcolors.LogNorm(vmin=abs_vmin, vmax=abs_vmax)

    a_vals = sorted(set(k[0] for k in results.keys()))
    c_vals = sorted(set(k[1] for k in results.keys()))

    # --- Individual plots: local norms per panel ---
    for (a, c), data in results.items():
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        raw = data["raw"]
        nz  = raw[raw != 0]
        lt  = np.percentile(np.abs(nz), 10) if len(nz) else 1.0
        local_norm_raw = mcolors.SymLogNorm(
            linthresh=lt, vmin=-np.max(np.abs(raw)), vmax=np.max(np.abs(raw))
        )

        pos = data["abs"][data["abs"] > 0]
        local_norm_abs = mcolors.LogNorm(
            vmin=np.percentile(pos, 5)  if len(pos) else 1e-10,
            vmax=np.percentile(pos, 95) if len(pos) else 1.0,
        )

        sns.heatmap(raw, xticklabels=d_vals, yticklabels=b_vals,
                    ax=axes[0], cmap="vlag", norm=local_norm_raw)
        axes[0].set_title(f"PCIC expected bias (a={a}, c={c})")

        sns.heatmap(data["abs"], xticklabels=d_vals, yticklabels=b_vals,
                    ax=axes[1], cmap=ABS_CMAP, norm=local_norm_abs)
        axes[1].set_title(f"PCIC expected absolute bias (a={a}, c={c})")

        plt.tight_layout()
        plt.savefig(f"{out_dir}/pcic_a{a}_c{c}.png", dpi=200)
        plt.close()

    # --- Grid plots: single shared colorbar ---
    def plot_grid(key, title, cmap, norm):
        fig, axes = plt.subplots(len(a_vals), len(c_vals), figsize=(18, 18))

        for i, a in enumerate(a_vals):
            for j, c in enumerate(c_vals):
                sns.heatmap(results[(a, c)][key], ax=axes[i, j],
                            cmap=cmap, cbar=False, norm=norm)
                axes[i, j].set_xticks([])
                axes[i, j].set_yticks([])
                if i == 0:
                    axes[i, j].set_title(f"c={c}", fontsize=10)
                if j == 0:
                    axes[i, j].set_ylabel(f"a={a}", fontsize=10)

        fig.suptitle(title)
        cbar = fig.colorbar(
            ScalarMappable(norm=norm, cmap=cmap),
            ax=axes.ravel().tolist(),
            shrink=0.85,
            aspect=25,
            pad=0.02,
        )
        cbar.ax.tick_params(labelsize=11)

        plt.savefig(f"{out_dir}/{key}_grid.png", dpi=300)
        plt.close()

    plot_grid("raw", "PCIC Expected Bias Grid",          "vlag",   norm_raw)
    plot_grid("abs", "PCIC Expected Absolute Bias Grid", ABS_CMAP, norm_abs)


plot_all(results, b_vals, d_vals)

In [12]:
# same but with var instead of abs
import numpy as np
import os
from scipy.special import logsumexp

def compute_expected_bias_vectorized(b, c, a=2, d=3, n_trials=2, n_train=100, n_test=200, n_grid=60):
    np.random.seed(42)

    Y_train = np.random.normal(0, 1, (n_trials, n_train))
    Y_test = np.random.normal(0, 1, (n_trials, n_test))

    t_vals = np.linspace(-1.5, 1.5, n_grid)
    theta1, theta2 = np.meshgrid(t_vals, t_vals)

    mean_s = (theta1.flatten() ** a) * (theta2.flatten() ** b)
    mean_t = (theta1.flatten() ** c) * (theta2.flatten() ** d)

    PCIC_trials = np.zeros(n_trials)
    G_n_trials = np.zeros(n_trials)

    for trial in range(n_trials):
        Y_tr = Y_train[trial]
        Y_te = Y_test[trial]

        log_s_train = -0.5 * ((Y_tr[:, None] - mean_s[None, :])**2) - 0.5 * np.log(2 * np.pi)

        log_post = np.sum(log_s_train, axis=0)
        log_post -= logsumexp(log_post)
        post_probs = np.exp(log_post)

        log_h_train = -0.5 * ((Y_tr[:, None] - mean_t[None, :])**2) - 0.5 * np.log(2 * np.pi)
        log_h_test  = -0.5 * ((Y_te[:, None] - mean_t[None, :])**2) - 0.5 * np.log(2 * np.pi)

        log_E_h_train = logsumexp(log_h_train + log_post[None, :], axis=1)
        log_E_h_test  = logsumexp(log_h_test  + log_post[None, :], axis=1)

        T_n = -np.mean(log_E_h_train)
        G_n = -np.mean(log_E_h_test)
        G_n_trials[trial] = G_n

        E_log_h_train   = np.sum(log_h_train * post_probs[None, :], axis=1)
        E_s_train       = np.sum(log_s_train * post_probs[None, :], axis=1)
        E_log_h_s_train = np.sum(log_h_train * log_s_train * post_probs[None, :], axis=1)

        pcic_pen = np.mean(E_log_h_s_train - E_log_h_train * E_s_train)
        PCIC_trials[trial] = T_n + pcic_pen

    errors = PCIC_trials - G_n_trials
    return np.mean(errors), np.var(errors)


def run_simulation_grid():
    a_values = [1, 2, 3, 4, 5]
    c_values = [1, 2, 3, 4, 5]

    degrees = list(range(1, 19))
    n_deg = len(degrees)

    results = {}

    for a in a_values:
        for c in c_values:
            pcic_biases_raw = np.zeros((n_deg, n_deg))
            pcic_biases_var = np.zeros((n_deg, n_deg))

            for i, b_val in enumerate(degrees):
                for j, d_val in enumerate(degrees):
                    raw, var = compute_expected_bias_vectorized(
                        b=b_val, c=c, a=a, d=d_val
                    )
                    pcic_biases_raw[i, j] = raw
                    pcic_biases_var[i, j] = var

            results[(a, c)] = {"raw": pcic_biases_raw, "var": pcic_biases_var}

    return results, degrees, degrees

results, b_vals, d_vals = run_simulation_grid()

In [13]:
# slightly updated plotting code to account for change to variance
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
import numpy as np
import seaborn as sns
import os

ABS_CMAP = "YlOrRd"

def plot_all(results, b_vals, d_vals, out_dir="./abcd"):
    os.makedirs(out_dir, exist_ok=True)

    all_raw = np.concatenate([v["raw"].ravel() for v in results.values()])
    all_var = np.concatenate([v["var"].ravel() for v in results.values()])

    # --- Global norms for grid plots ---
    nonzero_raw = all_raw[all_raw != 0]
    linthresh   = np.percentile(np.abs(nonzero_raw), 10) if len(nonzero_raw) else 1.0
    raw_vabs    = np.percentile(np.abs(all_raw), 95)
    norm_raw    = mcolors.SymLogNorm(linthresh=linthresh, vmin=-raw_vabs, vmax=raw_vabs)

    pos_var  = all_var[all_var > 0]
    var_vmin = np.percentile(pos_var, 5)  if len(pos_var) else 1e-10
    var_vmax = np.percentile(pos_var, 95) if len(pos_var) else 1.0
    norm_var = mcolors.LogNorm(vmin=var_vmin, vmax=var_vmax)

    a_vals = sorted(set(k[0] for k in results.keys()))
    c_vals = sorted(set(k[1] for k in results.keys()))

    # --- Individual plots: local norms per panel ---
    for (a, c), data in results.items():
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        raw = data["raw"]
        nz  = raw[raw != 0]
        lt  = np.percentile(np.abs(nz), 10) if len(nz) else 1.0
        local_norm_raw = mcolors.SymLogNorm(
            linthresh=lt, vmin=-np.max(np.abs(raw)), vmax=np.max(np.abs(raw))
        )

        pos = data["var"][data["var"] > 0]
        local_norm_var = mcolors.LogNorm(
            vmin=np.percentile(pos, 5)  if len(pos) else 1e-10,
            vmax=np.percentile(pos, 95) if len(pos) else 1.0,
        )

        sns.heatmap(raw, xticklabels=d_vals, yticklabels=b_vals,
                    ax=axes[0], cmap="vlag", norm=local_norm_raw)
        axes[0].set_title(f"PCIC expected bias (a={a}, c={c})")

        sns.heatmap(data["var"], xticklabels=d_vals, yticklabels=b_vals,
                    ax=axes[1], cmap=ABS_CMAP, norm=local_norm_var)
        axes[1].set_title(f"PCIC error variance (a={a}, c={c})")

        plt.tight_layout()
        plt.savefig(f"{out_dir}/pcic_a{a}_c{c}.png", dpi=200)
        plt.close()

    # --- Grid plots: single shared colorbar ---
    def plot_grid(key, title, cmap, norm):
        fig, axes = plt.subplots(len(a_vals), len(c_vals), figsize=(18, 18))

        for i, a in enumerate(a_vals):
            for j, c in enumerate(c_vals):
                sns.heatmap(results[(a, c)][key], ax=axes[i, j],
                            cmap=cmap, cbar=False, norm=norm)
                axes[i, j].set_xticks([])
                axes[i, j].set_yticks([])
                if i == 0:
                    axes[i, j].set_title(f"c={c}", fontsize=10)
                if j == 0:
                    axes[i, j].set_ylabel(f"a={a}", fontsize=10)

        fig.suptitle(title)
        cbar = fig.colorbar(
            ScalarMappable(norm=norm, cmap=cmap),
            ax=axes.ravel().tolist(),
            shrink=0.85,
            aspect=25,
            pad=0.02,
        )
        cbar.ax.tick_params(labelsize=11)

        plt.savefig(f"{out_dir}/{key}_grid.png", dpi=300)
        plt.close()

    plot_grid("raw", "PCIC Expected Bias Grid",     "vlag",   norm_raw)
    plot_grid("var", "PCIC Error Variance Grid",    ABS_CMAP, norm_var)


plot_all(results, b_vals, d_vals)